# Verify the PyTorch projection correlation implementation

This notebook expects the project root to be the working directory, with `projcorr.py` and both CSV files in the same folder. In Colab, it automatically selects CUDA when a GPU is available.

In [6]:
import os
import sys
import time
import json
import glob
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Auto-detect Google Colab and mount Google Drive
if 'google.colab' in sys.modules:
    print("[Colab] Google Colab environment detected.")
    from google.colab import drive
    drive.mount('/content/drive')

    candidates = [
        '/content/drive/MyDrive/user/goit/Diplom/var1_Di/code/projCorr',
        '/content/drive/MyDrive/Diplom/var1_Di/code/projCorr',
        '/content/drive/MyDrive/var1_Di/code/projCorr',
        '/content/drive/MyDrive/code/projCorr'
    ]
    target_dir = None
    for cand in candidates:
        if os.path.isdir(cand):
            target_dir = cand
            break
    if target_dir:
        os.chdir(target_dir)
        if target_dir not in sys.path:
            sys.path.insert(0, target_dir)
        print(f"[Colab] Working directory set to: {os.getcwd()}")
else:
    current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
    if current_dir not in sys.path:
        sys.path.insert(0, current_dir)
    print(f"[Local] Working directory: {os.getcwd()}")

# Hardware accelerator check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


[Colab] Google Colab environment detected.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1
4
[Colab] Working directory set to: /content/drive/MyDrive/user/goit/Diplom/var1_Di/code/projCorr
2
Device: cuda
GPU: Tesla T4
VRAM: 15.64 GB


In [7]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import torch
from projcorr import pcor_test, ipcor_test, ipcor_rcv_test

x_path = Path('data_20261109_195329_x.csv')
y_path = Path('data_20261109_195329_y.csv')
assert Path('projcorr.py').exists(), f'projcorr.py was not found in {Path.cwd()}'
assert x_path.exists() and y_path.exists(), f'CSV files were not found in {Path.cwd()}'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Working directory: {Path.cwd()}')
print(f'Compute device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

x = pd.read_csv(x_path)
y = pd.read_csv(y_path)
n = len(x)
print(f'Data shapes: x={x.shape}, y={y.shape}')

Working directory: /content/drive/MyDrive/user/goit/Diplom/var1_Di/code/projCorr
Compute device: cuda
GPU: Tesla T4
Data shapes: x=(100, 5), y=(100, 5)


## 1. `pcor_test`

Run 999 permutations with `seed=1`. Expected values: T = 0.27747, PC = 0.4889854, p = 0.001.

In [8]:
res_pcor = pcor_test(x, y, n_perm=999, seed=1, device=device)
print(res_pcor)
assert abs(res_pcor.statistic['T'] - 0.27747) < 1e-4
assert abs(res_pcor.estimate['PC'] - 0.4889854) < 1e-5
assert abs(res_pcor.p_value - 0.001) < 1e-3
print('pcor_test matches the reference.')


	Projection correlation permutation test

data:  x and y
T = 0.27747, permutations = 999, p-value = 0.001
alternative hypothesis: true correlation is greater than 0
sample estimates:
          PC
   0.4889854

pcor_test matches the reference.


## 2. `ipcor_test`

Expected values: Z = 12.093, IPC = 0.4135423, p < 2.2 ? 10???.

In [9]:
res_ipcor = ipcor_test(x, y, device=device)
print(res_ipcor)
assert abs(res_ipcor.statistic['Z'] - 12.093) < 1e-3
assert abs(res_ipcor.estimate['IPC'] - 0.4135423) < 1e-5
assert res_ipcor.p_value < 1e-15
print('ipcor_test matches the reference.')


	Improved projection correlation asymptotic test

data:  x and y
Z = 12.093, p-value = < 2.2e-16
alternative hypothesis: true correlation is greater than 0
sample estimates:
         IPC
   0.4135423

ipcor_test matches the reference.


## 3. `ipcor_rcv_test`

Set the number of pairs to `ceil(log(n))`. To reproduce the reference R split exactly, pass the same `first_half` indices used in `test_projcorr.py`. Expected values: Z = 13.278, RCV T = 0.2655577.

In [10]:
first_half_r = [0, 1, 5, 6, 13, 17, 19, 20, 21, 23, 24, 27, 31, 32, 33, 34, 36, 37, 38, 39, 41, 42, 43, 44, 50, 53, 58, 63, 67, 68, 69, 71, 72, 73, 74, 78, 79, 81, 82, 83, 84, 86, 89, 90, 91, 93, 95, 96, 97, 98]
n_pairs = math.ceil(math.log(n))
res_rcv = ipcor_rcv_test(x, y, n_pairs=n_pairs, seed=1, first_half=first_half_r, device=device)
print(res_rcv)
assert abs(res_rcv.statistic['Z'] - 13.278) < 1e-3
assert abs(res_rcv.estimate['RCV T'] - 0.2655577) < 1e-5
assert res_rcv.p_value < 1e-15
print('ipcor_rcv_test matches the reference.')


	Improved projection correlation RCV asymptotic test

data:  x and y
Z = 13.278, p-value = < 2.2e-16
alternative hypothesis: true correlation is greater than 0
sample estimates:
       RCV T
   0.2655577

ipcor_rcv_test matches the reference.


## Summary

If all cells finish without an `AssertionError`, the three computed results match the reference within the specified tolerances. The `Compute device` line shows whether this Colab session used CUDA.